In [81]:
from faker import Faker
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import stats
import random

## **1. Création des profils d’utilisateurs**


On va génération un ensemblre de donnée avec faker ou on aura 
* movie  : [ # liste de dictionnaire de tous les films et qui vont avoir :  
    {  
    * 'title': 'Film_196'  
    * 'genre': 'Animation',  
    * 'year': 2010,  
    * 'duration': 84  
    }  
    ]

* user  : c'est une liste qui va stocker tous nos utilisateurs et les différentes informations sont
    * name : nom de l'utilisateur
    * email : email de connexion l'utisateur 
    * password : mot de pass de connexion 
    * age  : age de l'utilsateur (utile pour proposer un film en fonction de l'age)
    * preference : préférence de l'utilsateur
    * watch_history : [ #historique de visionnage  
        {  
            'movie' : nom du film  
            'genre' : genre du film  
            'rate'  : note du film 
            'date'  : date de visionnage
        }  

    ]


In [82]:
faker = Faker('fr_FR') #Langage par défaut en français


In [83]:
#On définit 10 genres
genres = [
    "Action",
    "Comédie",
    "Drame",
    "Horreur",
    "Science-Fiction",
    "Romance",
    "Animation",
    "Thriller",
    "Documentaire",
    "Aventure"
]

In [84]:
film = [] #liste contenant nos film

for i in range(200):
    film.append({
        "title": f"Film_{i+1}",
        "genre": random.choice(genres),
        "year": random.randint(1990, 2025),
        "duration": random.randint(80, 180),
        "rating" : random.randint(1,5)
    })


In [85]:
df_film = pd.DataFrame(film)

In [86]:
df_film.head()

,title,genre,year,duration,rating
0,Film_1,Aventure,2024,103,1
1,Film_2,Documentaire,1998,101,1
2,Film_3,Science-Fiction,1993,141,3
3,Film_4,Horreur,1992,117,2
4,Film_5,Romance,1999,144,4


In [87]:
users = [] #Liste contenant nos utilisateurs

id = 0
for i in range(50):

    age = random.randint(16, 70)

    # Préférences de l'utilisateur
    preferences = random.sample(
        genres,
        k=random.randint(1, 3)
    )

    # Historique de visionnage
    watch_history = []

    for j in range(random.randint(2, 15)):

        movie = random.choice(film)

        watch_history.append({
            "movie": movie["title"],
            "genre": movie["genre"],
            "rate": random.randint(1, 5),
            "date": faker.date_between(
                start_date="-2y",
                end_date="today"
            ).strftime("%Y-%m-%d")
        })
    id += 1 

    user = {
        "id" : id,
        "name": faker.name(),
        "email": faker.email(),
        "password": faker.password(
            length=10,
            special_chars=True,
            #digits=True,
            upper_case=True,
            lower_case=True
        ),
        "age": age,
        "preference": preferences,
        "watch_history": watch_history
    }

    users.append(user)

In [88]:
df_user = pd.DataFrame(users)
df_user = df_user.reset_index(drop=True) 

In [89]:
df_user.head()

,id,name,email,password,age,preference,watch_history
0,1,Thomas Guillet,thierrylopes@example.org,bdr19OjcO&,34,[Drame],"[{'movie': 'Film_126', 'genre': 'Thriller', 'r..."
1,2,Raymond Colas,marthemasse@example.net,abr+VLG5@4,25,"[Aventure, Drame]","[{'movie': 'Film_123', 'genre': 'Romance', 'ra..."
2,3,Raymond Pelletier,gabriel01@example.com,P&k6b3Cl@$,20,"[Animation, Drame]","[{'movie': 'Film_131', 'genre': 'Romance', 'ra..."
3,4,Suzanne Germain-Briand,pierrecaron@example.net,8_9Gf5WsUP,32,[Aventure],"[{'movie': 'Film_13', 'genre': 'Romance', 'rat..."
4,5,Andrée-Christiane Chauvin,bourgeoiscatherine@example.org,^xs@4ZFn@l,61,"[Science-Fiction, Thriller]","[{'movie': 'Film_160', 'genre': 'Thriller', 'r..."


## **2. Création du moteur de recommandation :**

### **2.1 Filtrer les films en fonction des préférences de l'utilisateur.**

In [100]:
df_user.loc[0, 'preference']

['Drame']

In [104]:
def preferences_films(preferences):
    films = df_film[df_film['genre'].isin(preferences)]
    return films

# Préférences du premier utilisateur
prefs = df_user.loc[0, 'preference']

print("Films conseillés :")
all_pref = preferences_films(prefs)

for movie in all_pref['title']:
    print(f"{movie}, ",end='')

Films conseillés :
Film_18, Film_20, Film_21, Film_25, Film_53, Film_60, Film_62, Film_87, Film_94, Film_99, Film_101, Film_141, Film_149, Film_164, Film_168, Film_185, 

* Recuperer les préférences de l'utilisateur connecté
* Créer un Dataframe qui va prendre 3 variables, le genre du film, les films et la note du film
* On va ajouter créer une fonction avec les préférences de l'utilisateur en paramètre, cette fonction va parcouris tout le dataset et va ressortir tous les films qui sont égales aux préférence de l'utilisateur

### **2.2 Classement des films en fonction des notes des utilisateurs ou des similitudes de genre**

#### **Classement selon les notes des utilisateurs**

* Utilisation de sort_values par rapport au 'rating' et comme c'est par la plus grande valeur, l'ascending = false

In [106]:
def ranking (df_film) :
    df_film_ranking = df_film.sort_values('rating',ascending=False)
    return df_film_ranking

df = ranking(df_film)
print('Voici la liste des films conseillé en fonction de la notation :')
for movie in df['title']:
    print(f"{movie}, ",end='')

Voici la liste des films conseillé en fonction de la notation :
Film_46, Film_64, Film_66, Film_39, Film_27, Film_20, Film_22, Film_21, Film_138, Film_140, Film_143, Film_167, Film_162, Film_163, Film_155, Film_153, Film_164, Film_199, Film_197, Film_125, Film_120, Film_123, Film_116, Film_113, Film_105, Film_111, Film_110, Film_109, Film_117, Film_118, Film_98, Film_90, Film_89, Film_88, Film_87, Film_195, Film_192, Film_175, Film_174, Film_169, Film_82, Film_77, Film_75, Film_86, Film_74, Film_48, Film_52, Film_47, Film_29, Film_32, Film_30, Film_56, Film_42, Film_68, Film_5, Film_6, Film_36, Film_26, Film_38, Film_60, Film_65, Film_176, Film_180, Film_181, Film_137, Film_161, Film_129, Film_31, Film_25, Film_16, Film_23, Film_11, Film_97, Film_100, Film_76, Film_73, Film_96, Film_80, Film_79, Film_104, Film_172, Film_171, Film_178, Film_187, Film_170, Film_144, Film_166, Film_159, Film_198, Film_196, Film_190, Film_188, Film_182, Film_142, Film_147, Film_168, Film_128, Film_135, Fil

#### **Classement selon la similitude de genre**

* Création d'une nouvelle variable 'similaire' qui va stocker soit 1 pour les préférences similaire ou 0 lorsqu'il sont différent
* Utilisation de la methode appy() qui va un fonctionner comme une boucle et parcourir chaque ligne de ligne de donnée pour le traiter
* Une fonction similarity() avec comme paramère le genre et renvoie 1 si deux genres sont pareil et renvoi 0 dans le cas contraire
* Puis on affiche nos données avec un sort_values en se basant sur la variable 'similaire'

#### **Recommandation**

* Création d'une fonction recommandation avec deux paramètre (utilisateur, films) qui va renvoyer une liste de recommandation à l'utilisateur en fonction du genre du dernier film regarder.  

Dans la fonction :
* On récupère l'historique de visionnage du l'utilisateur
* On récupère ensuite le dernier film regarder l'utilisateur
* On récupère ensuite son genre
* 

In [91]:
def recommend(user, movies):
    history = user["watch_history"]

    if not history:
        return []

    # Dernier film regardé
    last_movie = history[-1]

    # Genre du dernier film
    genre = last_movie["genre"]

    # Films déjà vus
    watched_movies = {
        movie["movie"]
        for movie in history
    }

    recommendations = [
        movie["title"]
        for movie in movies
        if movie["genre"] == genre
        and movie["title"] not in watched_movies
    ]

    return recommendations

In [92]:
# def my_supposion(preference):
#     print('Mes films proposés')
#     for film in df[preference].stack().tolist(): #transformation du nouveau dataset obtenue en list
#         print(film)

# my_supposion(pref)

In [ ]:
# 3. Analyse et visualisation

# Sélection d'un utilisateur
user = users[0]

# Transformation de l'historique en DataFrame
df_history = pd.DataFrame(user["watch_history"])

# Affichage de l'historique
print("Historique de visionnage de l'utilisateur :")
display(df_history)

# 1. Répartition des genres

genre_counts = df_history["genre"].value_counts()

plt.figure(figsize=(7, 7))

plt.pie(
    genre_counts,
    labels=genre_counts.index,
    autopct="%1.1f%%",
    startangle=90
)

plt.title("Répartition des genres visionnés par l'utilisateur")
plt.show()

# 2. Évolution des notes

df_history["date"] = pd.to_datetime(df_history["date"])
df_history = df_history.sort_values("date")

plt.figure(figsize=(10, 5))

plt.plot(
    df_history["date"],
    df_history["rate"],
    marker="o"
)

plt.title("Évolution des notes de l'utilisateur au fil du temps")
plt.xlabel("Date de visionnage")
plt.ylabel("Note donnée")
plt.grid(True)
plt.show()

# 3. Évaluations du mois dernier

last_date = df_history["date"].max()

df_last_month = df_history[
    df_history["date"] >= last_date - pd.Timedelta(days=30)
]

if df_last_month.empty:
    print("Aucune évaluation disponible pour le mois dernier.")
else:
    plt.figure(figsize=(10, 5))

    plt.plot(
        df_last_month["date"],
        df_last_month["rate"],
        marker="o"
    )

    plt.title("Évolution des notes de l'utilisateur au cours du mois dernier")
    plt.xlabel("Date de visionnage")
    plt.ylabel("Note donnée")
    plt.grid(True)
    plt.show()